#### Copyright 2022 Google LLC.

Licensed under the Apache License, Version 2.0 (the "License");

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

# https://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Android Management API - Quickstart

If you have not yet read the Android Management API Codelab we recommend that you do so before using this notebook. If you opened this notebook from the Codelab then follow the next instructions on the Codelab.

In order to run this notebook, you need:

*   An Android 6.0+ device.

## Setup

The base resource of your Android Management solution is a Google Cloud Platform project. All other resources (`Enterprises`, `Devices`, `Policies`, etc) belong to the project and the project controls access to these resources. A solution is typically associated with a single project, but you can create multiple projects if you want to restrict access to resources.

For this Codelab we have already created a project for you (project ID: `android-management-io-codelab`).

To create and access resources, you need to authenticate with an account that has edit rights over the project. The account running this Codelab has been given rights over the project above. To start the authentication flow, **run the cell below**.

To run a cell:

1. Click anywhere in the code block.
2. Click the &#9654; button in the top-left of the code block.

When you build a server-based solution, you should create a
[service account](https://developers.google.com/android/management/service-account)
so you don't need to authorize the access every time.

**Note on OAuth Credentials:** 
Before running the cell below, ensure you have created an **OAuth client ID** (Web application) in your Google Cloud Console. 
1. Go to the [Google Cloud Console](https://console.cloud.google.com/).
2. Navigate to **APIs & Services > OAuth consent screen** and finish configuring the consent screen.
3. Click **Audience** in the left navigation menu. Under the **Test users** section, click **Add users**, enter your email address in the side panel, and click **Save** (or click **Publish App** to push it to production).
4. Navigate to **APIs & Services > Credentials**.
5. Click **Create Credentials > OAuth client ID**, and choose **Web application**.
6. Set the **Authorized redirect URIs** to `https://google.github.io/android-management-api-samples/oauth_callback.html`.
7. You will need to paste your Client ID and Client Secret into the prompts when executing.

In [ ]:
import getpass
from apiclient.discovery import build
from google_auth_oauthlib.flow import Flow
from random import randint

client_id = input('Enter your Google Cloud OAuth Client ID: ')
client_secret = getpass.getpass('Enter your Google Cloud OAuth Client Secret: ')

CLIENT_CONFIG = {
  'web': {
    'client_id': client_id,
    'auth_uri':'https://accounts.google.com/o/oauth2/auth',
    'token_uri':'https://oauth2.googleapis.com/token',
    'auth_provider_x509_cert_url':'https://www.googleapis.com/oauth2/v1/certs',
    'client_secret': client_secret
  }
}
SCOPES = ['https://www.googleapis.com/auth/androidmanagement']
CALLBACK_URL = 'https://google.github.io/android-management-api-samples/oauth_callback.html'

# Run the OAuth flow.
flow = Flow.from_client_config(CLIENT_CONFIG, SCOPES)
flow.redirect_uri = CALLBACK_URL
auth_url, _ = flow.authorization_url()
print('Please visit this URL to authorize this application: {}'.format(auth_url))

code = input('Enter the authorization code: ')
flow.fetch_token(code=code)

# Create the API client.
androidmanagement = build('androidmanagement', 'v1', credentials=flow.credentials)

print('\nAuthentication succeeded.')

## Select an enterprise

An **`Enterprise`** resource binds an organization to your Android Management solution.
**`Devices`** and **`Policies`** both belong to an enterprise. Typically, a single enterprise
resource is associated with a single organization. However, you can create multiple
enterprises for the same organization based on their needs. For example, an
organization may want separate enterprises for its different departments or regions.

For this Codelab we have already created an enterprise for you. **Run the next cell** to select it.

In [ ]:
enterprise_name = 'enterprises/LC02de1hmx'

## Create a policy

A **`Policy`** is a group of settings that determine the behavior of a managed device
and the apps installed on it. Each Policy resource represents a unique group of device
and app settings and can be applied to one or more devices. Once a device is linked to
a policy, any updates to the policy are automatically applied to the device.

To create a basic policy, **run the cell below**. You'll see how to create more advanced policies later in this guide.

In [ ]:
import json

# Create a random policy name to avoid colision with other Codelabs
if 'policy_name' not in locals():
  policy_name = enterprise_name + '/policies/' + str(randint(1, 1000000000))

policy_json = '''
{
  "applications": [
    {
      "packageName": "com.google.samples.apps.iosched",
      "installType": "FORCE_INSTALLED"
    }
  ],
  "advancedSecurityOverrides": {
     "developerSettings": "DEVELOPER_SETTINGS_ALLOWED"
  }
}
'''

androidmanagement.enterprises().policies().patch(
    name=policy_name,
    body=json.loads(policy_json)
).execute()

## Provision a device

Provisioning refers to the process of enrolling a device with an enterprise, applying the appropriate policies to the device, and guiding the user to complete the set up of their device in accordance with those policies. Before attempting to provision a device, ensure that the device is running Android 6.0 or above.

You need an enrollment token for each device that you want to provision (you can use the same token for multiple devices), when creating a token you can specify a policy that will be applied to the device.

In [ ]:
enrollment_token = androidmanagement.enterprises().enrollmentTokens().create(
    parent=enterprise_name,
    body={"policyName": policy_name}
).execute()

Embed your enrollment token in either an enrollment link or a QR code, and then follow the provisioning instructions below.

In [ ]:
from IPython.display import display, HTML

qr_code_content = enrollment_token['qrCode']
qr_code_settings = {
  'text': qr_code_content,
  'width': 500,
  'height': 500
}

display(HTML('''
  <div id="qr_code" style="padding: 16px; background: #FFF; width: 500px;"></div>
  <script type="text/javascript" src="//cdn.rawgit.com/davidshimjs/qrcodejs/gh-pages/qrcode.min.js"></script>
  <script type="text/javascript">
    new QRCode(document.getElementById("qr_code"), {});
  </script>
'''.format(qr_code_settings)))

In [ ]:
enrollment_link = 'https://enterprise.google.com/android/enroll?et=' + enrollment_token['value']

print('Please open this link on your device:', enrollment_link)

The method for provisioning a device varies depending on the management mode you want to use.

### Fully managed mode

In **fully managed mode** the entire device is managed and the device needs to be factory reset before setup. To set up a device in fully managed mode you need to use a QR code.

For devices running Android 7.0 or above:

1.  Turn on a new or factory-reset device.
2.  Tap the same spot on the welcome screen six times to enter QR code mode.
3.  Connect to a WiFi network.
4.  Scan the QR code.

For devices running Android 6.0:

1.  Turn on a new or factory-reset device.
2.  Follow the setup wizard and enter your Wi-Fi details.
3.  When prompted to sign in, enter **afw#setup**.
4.  Tap Next, and then accept the installation of Android Device Policy.
5.  Scan the QR code.

### Work profile mode

In **work profile mode** corporate apps and data are kept secure in a self-contained work profile while the user keeps control of the rest of the device. To set up a work profile you can either use a QR code or an enrollment link.

Using the enrollment link:

1.  Make the link accessible on the device (send it via email or put it on a website).
2.  Open the link.

Or using the QR code:

1.  Go to **Settings** > **Google**.
2.  Tap "Set up your work profile".
3.  Scan the QR code.

## What's next?

By now you should have a managed device configured with a basic policy, but there's
much more you can do with the Android Management API.

First, we recommend
[**exploring the range of available policies**](https://developers.google.com/android/management/create-policy)
to build the right policy for your needs.  

Next, explore other features of the Android Management API:

*   Learn how to [discover apps](https://developers.google.com/android/management/apps)
*   Set up [Pub/Sub notifications](https://developers.google.com/android/management/notifications)

Or start developing a server-based solution:

*   Download the Android Management API client library for
    [Java](https://developers.google.com/api-client-library/java/apis/androidmanagement/v1),
    [.NET](https://developers.google.com/api-client-library/dotnet/apis/androidmanagement/v1),
    [Python](https://developers.google.com/api-client-library/python/apis/androidmanagement/v1),
    or [Ruby](https://developers.google.com/api-client-library/ruby/apis/androidmanagement/v1).
*   Create a [service account](https://developers.google.com/android/management/service-account).

